# Batch evaluation & scoring

Purpose
-------

- Run batch inference on a curated eval split from manifest;
- Compute simple objective metrics (WER if ASR available), duration stats, and produce a short report;
- Save generated audio and JSON report for manual inspection and ranking;
- Keep evaluation reproducible and small (num_eval_samples is configurable).

In [ ]:
import json
import random
import time
from pathlib import Path
from statistics import mean
from typing import Dict, List, Optional

import soundfile as sf
from TTS.api import TTS

project_root = Path(".")
manifest_path = project_root / "data/processed/manifests/smoke_manifest.jsonl"
outputs_dir = project_root / "outputs/smoke_vits"
eval_dir = outputs_dir / "eval_batch"
eval_wavs_dir = eval_dir / "wavs"
report_path = eval_dir / "evaluation_report.json"

eval_dir.mkdir(parents=True, exists_ok=True)
eval_wavs_dir.mkdir(parents=True, exists_ok=True)

num_eval_samples = 20
use_local_checkpoint = True
public_model_name = "tts_models/pt/cv/vits"
random_seed = 42

use_asr_for_wer = True

## Prepare evaluation examples

- Sample up to `num_eval_samples` entries from the manifest (balanced or random);
- Save a small JSON with the evaluation examples for reproducibility.

In [ ]:
def load_manifest(manifest: Path) -> List[Dict]:
    entries: List[Dict] = []

    if not manifest.exists():
        return entries

    with open(manifest, "r", encoding="utf-8") as file:
        for line in file:
            try:
                entries.append(json.loads(line))
            except Exception:
                continue

    return entries


random.seed(random_seed)
all_entries = load_manifest(manifest_path)

if not all_entries:
    raise FileNotFoundError(f"No manifest entries found at {manifest_path}")

sampled_entries = (
    all_entries[:num_eval_samples]
    if len(all_entries) <= num_eval_samples
    else random.sample(all_entries, num_eval_samples)
)
sample_json = eval_dir / "sampled_entries.json"

with open(sample_json, "w", encoding="utf-8") as file:
    json.dump(sampled_entries, file, ensure_ascii=False, indent=2)

print(f"Prepared {len(sampled_entries)} evaluation examples -> {sample_json}")

## Batch inference

- Load a model (local checkpoint copy preferred, fallback to a public model);
- Synthesize each sampled example and write WAVs to `eval_wavs_dir`;
- Record synthesis time per sample.

In [ ]:
local_checkpoint_candidates = list((eval_dir).glob("*.*"))
model_to_use: Optional[str] = None

if use_local_checkpoint and local_checkpoint_candidates:
    checkpoint = next(
        (
            part
            for part in local_checkpoint_candidates
            if part.suffix in {".pth", ".pt", ".ckpt", ".tar", ".tar.gz"}
        ),
        None,
    )

    if checkpoint:
        model_to_use = str(checkpoint.resolve())
        print("Using local checkpoint for inference:", model_to_use)

if not model_to_use:
    model_to_use = public_model_name
    print("Using public prebuilt model for inference:", model_to_use)

tts = TTS(model_name=model_to_use)
sample_rate = getattr(getattr(tts, "synthesizer", None), "output_sample_rate", 22050)

evaluation_results = []

for idx, rec in enumerate(sampled_entries, start=1):
    text = rec.get("text") or rec.get("transcript") or ""

    if not text:
        continue

    start_ts = time.time()
    wav_array = tts.tts(text)
    elapsed = time.time() - start_ts
    output_path = eval_wavs_dir / f"eval_{idx:03d}.wav"
    sf.write(str(output_path, wav_array, sample_rate))

    evaluation_results.append(
        {
            "idx": idx,
            "wav_path": str(output_path),
            "reference_text": text,
            "duration_s": (
                round(len(wav_array) / sample_rate, 3)
                if hasattr(wav_array, "__len__")
                else None
            ),
            "synthesis_time_s": round(elapsed, 3),
        }
    )

    print(
        f"Wrote {output_path} (dur={evaluation_results[-1]['duration_s']}, synth_s={elapsed:.2f})"
    )

## Optional transcription + WER

- If `use_asr_for_wer` is `True` and `whisper` is available, transcribe generated WAVs with a smal ASR model and compute WER using jiwer;
- If ASR is not available, the notebook skips WER and still reports duration and synthesis timing.

In [ ]:
wer_value: Optional[float] = None

try:
    if use_asr_for_wer:
        import importlib

        whisper_spec = importlib.util.find_spec("whisper")

        if whisper_spec is None:
            print(
                'whisper not installed - skipping ASR/WER. Install with "pip install -U openai-whisper"'
            )
        else:
            import whisper
            from jiwer import wer

            asr_model = whisper.load_model("small")
            hypotheses = []
            references = []

            for item in evaluation_results:
                wav_path = item["wav_path"]
                result = asr_model.transcribe(wav_path, language="pt", verbose=False)
                hyphothesis = result.get("text", "").strip()
                hypotheses.append(hyphothesis)
                references.append(item["reference_text"].strip())

            wer_value = wer(references, hypotheses)
            print(f"Computed WER over {len(hypotheses)} samples: {wer_value:.3f}")

except Exception as error:
    print("ASR/WER step failed or skipped:", error)
    wer_value = None

## Metrics & sport

- Compute duration stats, mean synthesis time, and include optional WER;
- Save a JSON report with per-sample records and aggregate metrics.

In [ ]:
durations = [
    result.get("duration_s")
    for result in evaluation_results
    if result.get("duration_s") is not None
]
synthesis_times = [
    result.get("synthesis_time_s")
    for result in evaluation_results
    if result.get("synthesis_time_s") is not None
]

report = {
    "generated_count": len(evaluation_results),
    "mean_duration_s": round(mean(durations), 3) if durations else None,
    "min_duration_s": round(min(durations), 3) if durations else None,
    "max_duration_s": round(max(durations), 3) if durations else None,
    "mean_synthesis_time_s": (
        round(mean(synthesis_times), 3) if synthesis_times else None
    ),
    "wer": round(wer_value, 4) if wer_value is not None else None,
    "samples": evaluation_results,
}

with open(report_path, "w", enconding="utf-8") as file:
    json.dump(report, file, ensure_ascii=False, indent=2)

print(f"Saved evaluation report -> {report_path}")
print(
    "Summary:",
    {
        key: report[key]
        for key in (
            "generated_count",
            "mean_duration_s",
            "mean_synthesis_time_s",
            "wer",
        )
    },
)